# Unit 10 solutions: approximate nearest neighbours (ANN) and hybrid retrieval

These reference solutions build the **retrieval infrastructure** this unit is about -- an approximate index over an existing dense model -- and read every number honestly. Every dense path the course has built (Unit 7 semantic, Unit 8 two-tower, Unit 9 feature tower) retrieves the same exact way: score all 2,000 books for a reader by the `reader . item` dot product and sort. That is linear in the catalog. Here I build a FAISS **HNSW** graph over the Unit-8 two-tower item vectors, measure how closely it recovers the exact top-10 (**recall@10 vs exact**), and watch that recall rise with `efSearch`.

Two honest headlines frame the work, and the solutions state both plainly. First, **ANN preserves accuracy**: it recovers (approximately) the same top-10 the exact scan would, so its hit@10 equals the exact two-tower (~0.340). ANN buys **speed** at scale, not better recommendations -- and at 2,000 items there is no speedup at all; the win is asymptotic (the 20k Challenge shows recall falling at fixed `efSearch` as the catalog grows). Second, the **hybrid** path fuses Unit-3 BM25 with the two-tower: the tuned dense-heavy fusion (`pool=50, weight=0.7`) edges the two-tower on both hit@10 (~0.362) and coverage (~0.192 vs 0.177), but the lift is small and tuning-dependent -- equal weights *hurt* and reciprocal-rank fusion *loses*, which the Challenge records rather than gates.

Everything is measured on the **validation** split; each reader's **seen** set is their train-positive history; no cell reads the sealed `test` split. The index is built single-threaded (`faiss.omp_set_num_threads(1)`, saved and restored inside `HnswIndex`) and ranks by inner product, so it ranks the same dot product the two-tower's exact scan uses. The notebook runs top-to-bottom with fixed seeds and keeps its executed outputs; the two-tower is fitted **once** in the setup cell (~15-19 s) and every exercise reuses it, and the HNSW builds are cheap (~0.2 s at 2k, ~3 s at 20k), so every cell stays well under the 120 s/cell budget.

In [1]:
import json
from collections import defaultdict

import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    AnnRetrievalPath,
    HnswIndex,
    HybridRetrievalPath,
    LexicalRetrievalPath,
    TwoTowerRetrievalPath,
    run_blended_scoreboard,
    run_validation_scoreboard,
)

SEED = 0
K = 10                  # every scoreboard reports at the top-10 cutoff
M = 32                  # HNSW graph degree (pinned)
EF_CONSTRUCTION = 200   # build-time graph exploration (pinned)
EF_SEARCH = 64          # query-time exploration knob (pinned: recall@10 vs exact ~0.999)

# --- the real interaction log + content (the same substrate as Units 1-9) ------------------
data_dir = bookrec.generated_dir()
catalog = bookrec.load_catalog(data_dir / "catalog.csv.gz")        # {item_id: Book} (2000 books)
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
keywords = bookrec.load_keywords(data_dir / "keywords.csv.gz")     # {item_id: "space joined tokens"}

cold = json.loads((data_dir / "cold_partitions.json").read_text())
cold_readers = set(cold["cold_readers"])      # readers the personalized paths cannot serve

# The interaction log as plain row dicts -- the row-mapping log the paths duck-type.
interactions = pd.read_csv(interactions_path)
rows = interactions.to_dict("records")

# Each reader's train-positive history is their `seen` (exclusion) set. We grade on `val`; no cell
# here reads the sealed `test` split.
seen_by = defaultdict(set)
for row in rows:
    if row["label"] == 1 and row["split"] == "train":
        seen_by[row["reader_id"]].add(row["item_id"])

# The Unit-8 two-tower is the dense SOURCE every path in this unit reuses -- ONE ~15-19 s fit.
# Its reader and item vectors live in ONE inner-product space (hit@10 ~0.340, the warm leader).
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
tt_art = two_tower.artifact()
item_matrix = tt_art["item_embeddings"]       # (2000, 32) float32 -- the vectors the index will hold
reader_matrix = tt_art["reader_embeddings"]   # (n_readers, 32) float32 -- the query vectors
item_ids = tt_art["item_ids"]                 # item id aligned to each item_matrix ROW
reader_ids = tt_art["reader_ids"]             # reader id aligned to each reader_matrix ROW

print(f"catalog books: {len(catalog_ids)}   fitted readers: {reader_matrix.shape[0]}")
print(f"item_matrix: {item_matrix.shape} {item_matrix.dtype}   reader_matrix: {reader_matrix.shape}")

catalog books: 2000   fitted readers: 540
item_matrix: (2000, 32) float32   reader_matrix: (540, 32)


## Exercise 1 - Build an HNSW index over the two-tower vectors and query it

The exact brute-force baseline is what every dense path already does: score every item by the `reader . item` dot product and sort. Below I write that `exact_top_k`, build the HNSW index over the **same** two-tower item matrix, query it for the first reader, and confirm the approximate neighbours recover almost all of the exact top-10. The one detail that must be right is the metric: `HnswIndex` builds `IndexHNSWFlat(dim, M, METRIC_INNER_PRODUCT)` so it ranks the same inner product the exact scan uses.

In [2]:
# The EXACT brute-force baseline: score every item by the reader.item dot product and sort.
# This is what every dense path already does -- linear in the catalog.
def exact_top_k(reader_vec, k):
    scores = reader_vec @ item_matrix.T
    order = np.argsort(-scores)                 # descending by score
    return [item_ids[row] for row in order[:k]]


# Build the approximate index over the SAME item vectors, ranking by inner product.
index = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)

one_row = 0                                     # the first fitted reader (a row into reader_matrix)
reader_vec = reader_matrix[one_row]

exact_ids = exact_top_k(reader_vec, K)          # the brute-force top-10 item ids
# index.search needs a 2-D batch of queries; give it one row via [np.newaxis, :].
labels, scores = index.search(reader_matrix[one_row][np.newaxis, :], K, efSearch=EF_SEARCH)
ann_ids = [item_ids[row] for row in labels[0] if row >= 0]   # ANN top-10 item ids (skip -1 pads)

index_size = index.size                         # 2000 indexed items
overlap = len(set(exact_ids) & set(ann_ids))    # exact top-10 the ANN recovered
mostly_agree = bool(overlap >= 9)               # ANN recovers almost all of the exact top-10 at ef=64

metric_explanation = (
    "The index must rank by METRIC_INNER_PRODUCT so it scores the SAME reader.item dot product the "
    "exact scan uses -- then 'recall vs exact' compares like with like. FAISS's default is L2 "
    "distance, which orders neighbours by a different geometry; under L2 the index would return a "
    "different top-10 and recall-vs-exact would measure a metric mismatch, not approximation error."
)

print(f"indexed items: {index_size}")
print(f"exact top-{K}: {exact_ids}")
print(f"ANN   top-{K}: {ann_ids}")
print(f"overlap with exact top-{K}: {overlap}/{K}  (mostly agree? {mostly_agree})")

assert index_size == 2000, "the index holds every one of the 2000 catalog items"
assert mostly_agree, "the ANN top-10 recovers almost all of the exact top-10 at the pinned efSearch"

indexed items: 2000
exact top-10: [296, 518, 1922, 651, 1324, 253, 834, 527, 157, 1429]
ANN   top-10: [296, 518, 1922, 651, 1324, 253, 834, 527, 157, 1429]
overlap with exact top-10: 10/10  (mostly agree? True)


## Exercise 2 - Recall@10 vs exact, and watch it rise with `efSearch`

How faithfully does the approximate index recover the exact top-10? **Recall@10 vs exact** is, per reader, the fraction of their exact top-10 the ANN search also returns, averaged over readers -- recall against the brute-force baseline, not against held-out relevance. The knob is `efSearch`: more exploration, higher recall, slower query. I sweep `8, 16, 32, 64, 128` and gate on the endpoints (recall can plateau near 1.0, so I do not demand strict per-step monotonicity), confirming the pinned `efSearch=64` already recovers essentially the whole exact top-10.

In [3]:
# Exact top-10 ROW indices for EVERY reader (the brute-force reference), in item_matrix row space.
def recall_vs_exact(idx, matrix, ef):
    """Mean per-reader overlap@10 of idx's ANN top-10 with the exact top-10 (same inner-product scan)."""
    exact_rows = np.argsort(-(reader_matrix @ matrix.T), axis=1)[:, :K]
    labels, _ = idx.search(reader_matrix, K, efSearch=ef)
    overlaps = []
    for i in range(reader_matrix.shape[0]):
        ann_rows = {int(r) for r in labels[i] if r >= 0}
        exact_set = {int(r) for r in exact_rows[i]}
        overlaps.append(len(ann_rows & exact_set) / K)
    return float(np.mean(overlaps))


ef_sweep = [8, 16, 32, 64, 128]
recall_by_ef = {ef: recall_vs_exact(index, item_matrix, ef) for ef in ef_sweep}
# Endpoints, not per-step (recall ties near 1.0); a tiny tolerance guards a float tie.
recall_rises = bool(recall_by_ef[8] <= recall_by_ef[128] + 1e-9)
recall_at_pinned = recall_by_ef[EF_SEARCH]      # ~0.999 at the pinned efSearch=64
pinned_is_high = bool(recall_at_pinned >= 0.98)

efsearch_explanation = (
    "efSearch controls how much of the navigable graph each query explores: a higher value visits "
    "more candidate neighbours, so recall rises toward the exact top-10 -- at a higher per-query "
    "cost. It trades query LATENCY for RECALL; the pinned 64 sits where recall is essentially 1.0 "
    "while the search is still cheap."
)

for ef in ef_sweep:
    print(f"efSearch={ef:>3}  recall@{K} vs exact = {recall_by_ef[ef]:.4f}")
print(f"pinned efSearch={EF_SEARCH}: recall {recall_at_pinned:.4f}  (>= 0.98? {pinned_is_high})")

assert recall_rises, "recall at the cheapest search does not exceed recall at the most thorough"
assert pinned_is_high, "the pinned efSearch=64 recovers essentially the whole exact top-10 (>= 0.98)"

efSearch=  8  recall@10 vs exact = 0.9813
efSearch= 16  recall@10 vs exact = 0.9963
efSearch= 32  recall@10 vs exact = 0.9983
efSearch= 64  recall@10 vs exact = 0.9991
efSearch=128  recall@10 vs exact = 0.9996
pinned efSearch=64: recall 0.9991  (>= 0.98? True)


## Exercise 3 - ANN preserves accuracy: hit@10 == the exact two-tower

Recall-vs-exact says the ANN neighbours *are* the exact neighbours almost every time, so wrapping the two-tower in an ANN path does not change its recommendations: its hit@10 on `val` is identical to the exact two-tower (~0.340). `AnnRetrievalPath` wraps the already-fit two-tower -- its `fit` just builds the HNSW index (the `interactions` argument is ignored), its `retrieve` answers by an approximate search. I score both paths and confirm the match.

In [4]:
# Wrap the two-tower: AnnRetrievalPath.retrieve approximates the exact scan via the HNSW index.
# fit ignores `interactions` (the two-tower is already trained); it just builds the index.
ann = AnnRetrievalPath(two_tower, efSearch=EF_SEARCH, m=M, ef_construction=EF_CONSTRUCTION).fit(rows)


def val_hit(path):
    return run_validation_scoreboard(
        path, interactions_path, catalog_ids=catalog_ids, k=K, cold_readers=cold_readers
    ).hit_rate_at_k


tt_hit = val_hit(two_tower)     # ~0.340 (the exact brute-force two-tower)
ann_hit = val_hit(ann)          # ~0.340 (identical -- ANN preserves accuracy)
ann_matches_exact = bool(abs(ann_hit - tt_hit) <= 0.005)

accuracy_explanation = (
    "ANN recovers (approximately) the SAME top-10 the exact scan would, so its hit@10 equals the "
    "two-tower's -- here the gap is 0. ANN is a SPEED technique: it buys sublinear query time at "
    "scale, not better recommendations. Reading a higher hit@10 into 'adding ANN' would be a "
    "mistake; the most an approximate index can do to accuracy is lose a sliver of it."
)

print(f"two-tower (exact) hit@{K} = {tt_hit:.3f}")
print(f"ANN (approx)      hit@{K} = {ann_hit:.3f}")
print(f"|ANN - exact| = {abs(ann_hit - tt_hit):.4f}  (within 0.005? {ann_matches_exact})")

assert ann_matches_exact, "ANN hit@10 matches the exact two-tower within tolerance (SPEED, not accuracy)"

two-tower (exact) hit@10 = 0.340
ANN (approx)      hit@10 = 0.340
|ANN - exact| = 0.0000  (within 0.005? True)


## Exercise 4 - The retrieve contract: over-fetch past `seen`, and unknown readers

A reader's `seen` set (train history) must be excluded -- a re-read is not a recommendation. If the index returned exactly `k` and then dropped the seen ones, a heavy reader would end up with fewer than `k` recs. So `AnnRetrievalPath.retrieve` **over-fetches** `min(n_items, k + len(seen))` neighbours *before* excluding `seen`. A reader with no dense vector returns `[]`. I confirm all three on the `ann` path from Exercise 3.

In [5]:
contract_reader = reader_ids[0]

empty_seen_recs = ann.retrieve(contract_reader, {"seen": set()}, K)
got_k = bool(len(empty_seen_recs) == K)

# Exclude 25 catalog items as this reader\'s "seen": none may appear, and we still get K recs.
heavy_seen = set(catalog_ids[:25])
heavy_recs = ann.retrieve(contract_reader, {"seen": heavy_seen}, K)
heavy_got_k = bool(len(heavy_recs) == K)
no_seen_leak = bool(all(c.item_id not in heavy_seen for c in heavy_recs))

# An unknown reader (no row in reader_matrix) returns [].
unknown_reader = next(r for r in range(10_000_000) if r not in set(reader_ids))
unknown_empty = bool(ann.retrieve(unknown_reader, {"seen": set()}, K) == [])

contract_explanation = (
    "Over-fetching k + len(seen) neighbours BEFORE excluding seen means a reader who has already "
    "read many of their top neighbours still has k UNSEEN candidates left to return -- exact parity "
    "with the brute-force scan, which would do the same. An unknown reader has no row in the reader "
    "matrix, so there is no query vector to search with and the path returns [] rather than guessing."
)

print(f"empty seen: {len(empty_seen_recs)} recs (== K? {got_k})")
print(f"25 excluded: {len(heavy_recs)} recs (== K? {heavy_got_k})  no excluded id leaked? {no_seen_leak}")
print(f"unknown reader {unknown_reader}: returns []? {unknown_empty}")

assert got_k, "a known reader with empty seen gets K recs"
assert heavy_got_k and no_seen_leak, "over-fetch keeps K recs past 25 seen, and none of them leak"
assert unknown_empty, "an unknown reader has no dense vector, so the path returns []"

empty seen: 10 recs (== K? True)
25 excluded: 10 recs (== K? True)  no excluded id leaked? True
unknown reader 13: returns []? True


## Exercise 5 - Determinism: two single-thread builds give the same index

HNSW construction is thread-order dependent, so `HnswIndex` builds single-threaded (`faiss.omp_set_num_threads(1)`, saved and restored in a `finally` block) and two builds over the same matrix give the same approximate neighbours. The gate (design 011's determinism law) is **identical neighbour ids + `allclose` scores**, not exact-float equality (FAISS returns float32) and not exact identity against the brute force (ANN is approximate by construction). I build two indices, query both, and confirm they agree.

In [6]:
# Two independently-built single-thread indices over the SAME item matrix.
first = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)
second = HnswIndex(item_matrix, item_ids, m=M, ef_construction=EF_CONSTRUCTION)

labels_a, scores_a = first.search(reader_matrix, K, efSearch=EF_SEARCH)
labels_b, scores_b = second.search(reader_matrix, K, efSearch=EF_SEARCH)

ids_identical = bool(np.array_equal(labels_a, labels_b))    # the gate: same neighbour ids
scores_allclose = bool(np.allclose(scores_a, scores_b))     # the gate: float32 scores agree

determinism_explanation = (
    "A single-thread build fixes the insertion order and so fixes the graph, making two builds "
    "return identical neighbour ids. We gate on identical ids + allclose scores (not exact floats, "
    "since FAISS returns float32 inner products) -- and NOT on matching the brute-force top-10, "
    "because ANN is approximate by design; Exercises 1-2 already measured that approximation."
)

print(f"two builds: identical neighbour ids? {ids_identical}")
print(f"two builds: scores allclose?         {scores_allclose}")

assert ids_identical, "two single-thread builds give identical neighbour ids (design 011 determinism)"
assert scores_allclose, "and allclose float32 scores -- reproducible, the determinism gate"

two builds: identical neighbour ids? True
two builds: scores allclose?         True


## Exercise 6 - A hybrid path: fuse BM25 + the two-tower, read hit@10 AND coverage

BM25 (sparse keyword overlap) and the two-tower (dense taste) surface largely different books. `HybridRetrievalPath` fuses them: each over-fetches its top `pool`, each path's scores are min-max normalised over its own pool, and the fused score is `weight * dense + (1 - weight) * sparse`. The pinned `weight=0.7` is dense-heavy because the dense tower is stronger. I score BM25-alone, the two-tower-alone, and the hybrid on both hit@10 and coverage, and read the honest story: dense dominates, the tuned hybrid gives a small lift on both axes, not a free win.

In [7]:
# Unit 3\'s BM25 lexical path (sparse): a reader\'s query is the words of the books they have read.
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)

# The pinned dense-heavy hybrid: fuse BM25 + the two-tower over each path\'s top-50 pool (w_dense=0.7).
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50, method="weighted").fit(rows)


def board(path):
    """hit@10 + catalog coverage via the single-path blended scoreboard (reproduces the path\'s hit@k)."""
    result = run_blended_scoreboard(
        {path.name: path}, interactions_path,
        catalog_ids=catalog_ids, k=K, pool=K, cold_readers=cold_readers,
    )
    return result.hit_rate_at_k, result.catalog_coverage


bm25_hit, bm25_cov = board(lexical)      # ~0.158 / ~0.278
tt_hit2, tt_cov = board(two_tower)       # ~0.340 / ~0.177
hybrid_hit, hybrid_cov = board(hybrid)   # ~0.362 / ~0.192

dense_dominates = bool(tt_hit2 > bm25_hit)             # the dense tower is the stronger path
hybrid_edges_hit = bool(hybrid_hit >= tt_hit2 - 0.01)  # directional, measured-safe (NOT strict >)
hybrid_lifts_coverage = bool(hybrid_cov > tt_cov)

hybrid_reading = (
    "The ordering is two-tower (~0.340) >> BM25 (~0.158): the dense tower dominates. The tuned "
    "dense-heavy hybrid edges the two-tower on BOTH axes -- hit@10 ~0.362 and coverage ~0.192 vs "
    "0.177 -- but the lift is SMALL and tuning-dependent, not a free win (equal weights and RRF "
    "lose, Exercise 8). BM25 contributes lexically on-the-nose matches the dense tower ranks low, "
    "which is what widens catalog coverage; it is a modest complement, not a new best."
)

print(f"BM25      hit@{K}/coverage = {bm25_hit:.3f} / {bm25_cov:.3f}")
print(f"two-tower hit@{K}/coverage = {tt_hit2:.3f} / {tt_cov:.3f}")
print(f"hybrid    hit@{K}/coverage = {hybrid_hit:.3f} / {hybrid_cov:.3f}")

assert dense_dominates, "the dense two-tower is the stronger single path (dominates BM25)"
assert hybrid_edges_hit, "the tuned hybrid holds hit within 0.01 of the two-tower (directional)"
assert hybrid_lifts_coverage, "the tuned hybrid lifts catalog coverage above the two-tower"

BM25      hit@10/coverage = 0.158 / 0.278
two-tower hit@10/coverage = 0.340 / 0.177
hybrid    hit@10/coverage = 0.362 / 0.192


## Exercise 7 - Challenge: recall across the catalog, and why speed is asymptotic

At a *fixed* `efSearch` recall **falls** as the catalog grows -- a bigger graph needs more exploration to find the true top-10. And the speed win is asymptotic: at 2,000 items brute force over `2000 x 32` is already microseconds, but at tens of thousands of items the graph visits only a handful of nodes while brute force must touch every item. I build a 20k synthetic index (ten lightly-jittered, seeded copies of the item matrix), measure recall@10 vs exact at the pinned `efSearch=64`, and confirm it is lower than the 2k recall at the same `efSearch`.

In [8]:
# A 20k synthetic catalog: ten lightly-jittered copies of the 2000 item vectors (seeded).
# HnswIndex coerces to contiguous float32 internally, so we can pass the stacked array directly.
rng = np.random.default_rng(SEED)
copies = [item_matrix + 0.01 * rng.normal(size=item_matrix.shape) for _ in range(10)]
big_matrix = np.concatenate(copies)             # (20000, 32)
big_ids = list(range(big_matrix.shape[0]))
big_index = HnswIndex(big_matrix, big_ids, m=M, ef_construction=EF_CONSTRUCTION)

big_recall = recall_vs_exact(big_index, big_matrix, EF_SEARCH)   # ~0.99 at 20k
small_recall = recall_by_ef[EF_SEARCH]          # ~0.999 at 2k (reuse Exercise 2)
recall_falls_with_size = bool(big_recall < small_recall)   # same efSearch, bigger catalog
big_index_size = big_index.size                 # 20000 indexed items

scaling_explanation = (
    "At a FIXED efSearch recall FALLS as the catalog grows (here ~0.999 at 2k -> lower at 20k): a "
    "bigger graph needs more exploration to reach the true neighbours, so efSearch must scale with "
    "the catalog. The SPEED win is asymptotic -- at 2k brute force over 2000x32 is already "
    "microseconds and HNSW is no faster, but at tens of thousands of items the graph visits only a "
    "handful of nodes while brute force touches every item. That asymptotic regime is exactly when "
    "an approximate index earns its place."
)

print(f"20k index: {big_index_size} items, recall@{K} vs exact = {big_recall:.4f}")
print(f" 2k index: recall@{K} vs exact = {small_recall:.4f}")
print(f"recall falls at fixed efSearch as the catalog grows? {recall_falls_with_size}")

assert big_index_size == 20000, "the stacked synthetic catalog holds 20k items"
assert recall_falls_with_size, "at a fixed efSearch, recall falls as the catalog grows 2k -> 20k"

20k index: 20000 items, recall@10 vs exact = 0.9880
 2k index: recall@10 vs exact = 0.9991
recall falls at fixed efSearch as the catalog grows? True


## Exercise 8 - Challenge: fusion is not free -- equal weights hurt and RRF loses

The Exercise-6 hybrid used one tuned setting. Is fusion *generally* a win? No. Two natural alternatives both **lose** to the two-tower alone: **equal weights** (`weight=0.5`, giving weak BM25 as much say as the dense tower) and **reciprocal-rank fusion** (`method="rrf"`, fusing by rank, not normalised score). Only the tuned dense-heavy weighted sum gives the small lift. I score all three against the two-tower-alone hit@10 (`tt_hit2` from Exercise 6). Because these comparisons are weight-sensitive, the honest readout is **recorded (printed)**: only the tuned lift is directionally gated.

In [9]:
def hybrid_hit(weight, pool, method):
    """hit@10 of a hybrid at the given fusion settings (reuse board() from Exercise 6)."""
    path = HybridRetrievalPath(lexical, two_tower, weight=weight, pool=pool, method=method).fit(rows)
    hit, _coverage = board(path)
    return hit


pinned_hit = hybrid_hit(0.7, 50, "weighted")   # ~0.362 (tuned dense-heavy; the lift)
equal_hit = hybrid_hit(0.5, 50, "weighted")    # ~0.31  (equal weights HURT)
rrf_hit = hybrid_hit(0.5, 50, "rrf")           # ~0.33  (RRF loses)

# Reuse tt_hit2 (the two-tower-alone hit@10) from Exercise 6.
equal_weights_hurt = bool(equal_hit < tt_hit2)
rrf_loses = bool(rrf_hit < tt_hit2)
only_tuned_helps = bool(pinned_hit >= tt_hit2 - 0.01 and equal_weights_hurt and rrf_loses)

fusion_explanation = (
    "Equal weights and standard RRF both come in BELOW the two-tower alone; only the tuned "
    "dense-heavy weight (0.7) holds the small lift. Fusing two score distributions on very "
    "different scales needs min-max normalisation PLUS a tuned weight that respects which path is "
    "stronger: equal weights hand the weak BM25 path too much say, and RRF throws away the "
    "score magnitudes entirely (it fuses by rank), so both drag the strong dense ranking down. "
    "Naive fusion is not safe -- these readings are weight-sensitive, so they are recorded, not gated."
)

print(f"two-tower alone  hit@{K} = {tt_hit2:.3f}")
print(f"pinned w=0.7     hit@{K} = {pinned_hit:.3f}  (holds the lift)")
print(f"equal w=0.5      hit@{K} = {equal_hit:.3f}  (hurt? {equal_weights_hurt})")
print(f"RRF              hit@{K} = {rrf_hit:.3f}  (loses? {rrf_loses})")
print(f"only the tuned fusion helps? {only_tuned_helps}")

# Directionally GATED (measured-safe): the tuned fusion holds its lift. The equal-weights / RRF
# losses above are RECORDED (printed), not asserted -- they are weight-sensitive, not a secure gate.
assert pinned_hit >= tt_hit2 - 0.01, "only the tuned dense-heavy weighted fusion holds its lift"

two-tower alone  hit@10 = 0.340
pinned w=0.7     hit@10 = 0.362  (holds the lift)
equal w=0.5      hit@10 = 0.308  (hurt? True)
RRF              hit@10 = 0.328  (loses? True)
only the tuned fusion helps? True
